# MMS-TTS English (`facebook/mms-tts-eng`)

Run Meta's **Massively Multilingual Speech** English TTS on Kaggle.

| Item | Value |
|---|---|
| Model | [`facebook/mms-tts-eng`](https://huggingface.co/facebook/mms-tts-eng) |
| Family | [MMS-TTS (1100+ languages)](https://huggingface.co/facebook/mms-tts) |
| Architecture | **VITS** via Hugging Face `VitsModel` |
| Size | ~36.3M params (CPU is enough; GPU is faster) |
| Sample rate | 16 kHz |
| License | CC-BY-NC 4.0 (non-commercial) |

**Kaggle settings (required)**
1. Open this notebook on Kaggle
2. Right sidebar → **Settings** → **Internet: ON**
3. Accelerator: **GPU T4** if available, otherwise CPU works

Papers / code:
- MMS paper: [Scaling Speech Technology to 1,000+ Languages](https://arxiv.org/abs/2305.13516)
- VITS paper: [Conditional VAE with Adversarial Learning for End-to-End TTS](https://arxiv.org/abs/2106.06103)
- Meta repo: [fairseq/examples/mms](https://github.com/facebookresearch/fairseq/tree/main/examples/mms)

## 1. Install

`transformers >= 4.33` is required for `VitsModel`. Kaggle images are usually older, so upgrade first.

In [ ]:
import sys
print(sys.version)

!pip install -q --upgrade transformers accelerate scipy

import transformers, torch
print("transformers", transformers.__version__)
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. What is MMS-TTS? (architecture)

MMS trains **one separate VITS checkpoint per language**. English and Bangla are the same architecture, different weights + tokenizer vocab.

VITS = **V**ariational **I**nference with adversarial learning for end-to-end **T**ext-to-**S**peech. It is a **conditional VAE** that maps text → waveform in one stage (no separate spectrogram model + vocoder pipeline).

### Modules (what `VitsModel` actually contains)

| Module | Class | Role | ~params on eng |
|---|---|---|---|
| `text_encoder` | `VitsTextEncoder` | Transformer over characters → hidden text features + prior stats | 6.3M |
| `duration_predictor` | `VitsStochasticDurationPredictor` | How long each character is spoken (rhythm). Stochastic → same text, different timing | 1.3M |
| `flow` | `VitsResidualCouplingBlock` | Normalizing flow on the latent (expressiveness) | 7.1M |
| `decoder` | `VitsHifiGan` | Latent/spectrogram → 16 kHz waveform (HiFi-GAN-style transposed convs) | 14.3M |
| `posterior_encoder` | `VitsPosteriorEncoder` | **Training only**: encodes real audio into the latent `z` | 7.2M |

### Inference path (this notebook)

```
text  →  tokenizer (characters + blank tokens)
      →  text_encoder
      →  stochastic duration predictor  (upsample each char to frames)
      →  sample latent z from prior + flow
      →  HiFi-GAN decoder
      →  waveform @ 16 kHz
```

The posterior encoder is unused at inference. Because duration and latent sampling are random, **set a seed** if you need the same audio twice.

### Training (for later / Bangla finetune context)

Trained end-to-end with:
- VAE ELBO / KL between posterior and flow-transformed prior
- adversarial loss on the waveform (GAN discriminator lives in the *full* fairseq checkpoint, not in this HF generator-only card)
- reconstruction on the spectrogram

Default VITS sizes used by MMS-TTS English:
- hidden 192, 6 Transformer layers, 2 heads, FFN 768
- spectrogram bins 513
- HiFi-GAN upsample rates `[8, 8, 2, 2]` → total hop **256**
- speaking_rate=1.0, noise_scale=0.667, noise_scale_duration=0.8

## 3. Load English checkpoint

In [ ]:
from transformers import VitsModel, VitsTokenizer, set_seed
import torch

MODEL_ID = "facebook/mms-tts-eng"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tokenizer = VitsTokenizer.from_pretrained(MODEL_ID)
model = VitsModel.from_pretrained(MODEL_ID).to(device).eval()

print("is_uroman (English should be False):", tokenizer.is_uroman)
print("sampling_rate:", model.config.sampling_rate)
print("params_M:", round(sum(p.numel() for p in model.parameters()) / 1e6, 2))

## 4. Inspect config + modules

Use this cell as evidence that the English checkpoint really is VITS.

In [ ]:
cfg = model.config
print("vocab_size", cfg.vocab_size)
print("hidden_size", cfg.hidden_size)
print("num_hidden_layers", cfg.num_hidden_layers)
print("num_attention_heads", cfg.num_attention_heads)
print("ffn_dim", cfg.ffn_dim)
print("use_stochastic_duration_prediction", cfg.use_stochastic_duration_prediction)
print("num_speakers", cfg.num_speakers)
print("upsample_rates", cfg.upsample_rates)
print("upsample_kernel_sizes", cfg.upsample_kernel_sizes)
print("prior_encoder_num_flows", cfg.prior_encoder_num_flows)
print("posterior_encoder_num_wavenet_layers", cfg.posterior_encoder_num_wavenet_layers)
print("speaking_rate", cfg.speaking_rate)
print("noise_scale", cfg.noise_scale)
print("noise_scale_duration", cfg.noise_scale_duration)
print()
print(f"{'module':22s} {'params':>8s}  class")
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"{name:22s} {n/1e6:7.2f}M  {type(module).__name__}")

print("\nvocab (id 0 is also used as the VITS blank between characters):")
print(sorted(tokenizer.get_vocab().items(), key=lambda kv: kv[1]))

## 5. Synthesize English speech

English uses a **Roman character tokenizer** (`is_uroman=False`). No extra romanization step.

The tokenizer inserts a **blank token** (id `0`) between characters. That is expected VITS / MMS behaviour, not a bug.

In [ ]:
text = "Hello, this is the English MMS text to speech model."
inputs = tokenizer(text, return_tensors="pt")
print("input_ids:", inputs["input_ids"].tolist())

set_seed(555)
with torch.no_grad():
    output = model(**{k: v.to(device) for k, v in inputs.items()})

waveform = output.waveform[0].detach().cpu()
seconds = waveform.numel() / model.config.sampling_rate
print(f"waveform shape={tuple(waveform.shape)}  min={waveform.min():.3f}  max={waveform.max():.3f}  seconds={seconds:.3f}")

In [ ]:
from IPython.display import Audio, display
import numpy as np
from scipy.io import wavfile

audio = waveform.numpy()
display(Audio(audio, rate=model.config.sampling_rate))

pcm16 = np.clip(audio * 32767, -32768, 32767).astype(np.int16)
wavfile.write("mms_tts_eng_sample.wav", model.config.sampling_rate, pcm16)
print("saved mms_tts_eng_sample.wav")

## 6. Optional: speaking rate / variation knobs

These are on `model.config` and affect the next forward pass.

In [ ]:
def speak(text, speaking_rate=1.0, noise_scale=0.667, noise_scale_duration=0.8, seed=555):
    model.config.speaking_rate = speaking_rate
    model.config.noise_scale = noise_scale
    model.config.noise_scale_duration = noise_scale_duration
    inputs = tokenizer(text, return_tensors="pt")
    set_seed(seed)
    with torch.no_grad():
        wav = model(**{k: v.to(device) for k, v in inputs.items()}).waveform[0].detach().cpu().numpy()
    return wav

fast = speak(text, speaking_rate=1.3)
print("faster seconds:", round(fast.size / model.config.sampling_rate, 3))
display(Audio(fast, rate=model.config.sampling_rate))

## Next: Bangla (`facebook/mms-tts-ben`)

Same `VitsModel` API. Difference to expect:

- Switch `MODEL_ID` to `facebook/mms-tts-ben`
- Check `tokenizer.is_uroman`. If `True`, Bengali script must be romanized (install `uroman`, Python >= 3.10) before / via the tokenizer
- Pass **Bengali text**, not English, into the Bangla checkpoint

Do not run Bangla in this notebook until English inference is confirmed on Kaggle.